# Autograd: PyTorch's automatic differentiation

_PyTorch (a complete course)_

**PyTorch records every operation, then computes all the gradients for you with one .backward() call.**

When you do math on a tensor that has requires_grad=True, PyTorch quietly records each operation into a computational graph &mdash; a directed graph of who-was-computed-from-whom. The graph is dynamic: it is built fresh, on the fly, as your Python code runs. Loops, if branches, varying shapes &mdash; all fine, because the graph is whatever your code actually did this time.

       Calling .backward() on a scalar output walks that graph backwards, applying the chain rule at each node, and deposits the gradient into every leaf's .grad. That backward walk is exactly reverse-mode automatic differentiation &mdash; which is exactly backpropagation (see dl-backprop).

---

This notebook is a practice scaffold for the **Autograd: PyTorch's automatic differentiation** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch

# --- 1. A leaf tensor we want gradients for ---
x = torch.tensor(2.0, requires_grad=True)

# --- 2. Compute something. PyTorch records a dynamic graph as we go. ---
y = x**3 + 2*x                      # y = 8 + 4 = 12
print("y          =", y.item())     # 12.0
print("y.grad_fn  =", y.grad_fn)    # <AddBackward0> : the graph edge that built y

# --- 3. Reverse-mode autodiff (backprop): fills x.grad ---
y.backward()
print("x.grad     =", x.grad.item())          # 14.0
print("analytic   =", (3*x.item()**2 + 2))    # 3*4 + 2 = 14.0  -> they match

# --- 4. GRADIENTS ACCUMULATE: backward ADDS into .grad ---
y2 = x**3 + 2*x
y2.backward()
print("after 2nd backward (no zero):", x.grad.item())  # 28.0  (14 + 14) -- the #1 bug!

# --- 5. The fix: zero the gradient before the next backward ---
x.grad.zero_()                      # in a model you'd call optimizer.zero_grad()
y3 = x**3 + 2*x
y3.backward()
print("after zero_grad + backward :", x.grad.item())   # 14.0  -- correct again

# --- 6. torch.no_grad(): stop tracking (inference / freezing) ---
with torch.no_grad():
    z = x**3 + 2*x                  # computed, but NO graph is built
print("z.requires_grad =", z.requires_grad)  # False -- saves memory at inference
print("z.grad_fn       =", z.grad_fn)         # None

# (.detach() does the same for a single tensor: x.detach() shares data but is graph-free)


## Visualize the data & results

_Does PyTorch's autograd actually compute the true derivative? Plot the autograd gradient of f(x)=x^3+2x against the analytic derivative 3x^2+2._

In [ ]:
import numpy as np
import torch

xs = np.linspace(-3, 3, 31)

# Analytic derivative of f(x) = x^3 + 2x  ->  3x^2 + 2
analytic = 3 * xs**2 + 2

# torch.autograd: differentiate the SAME function at each x and read x.grad
autograd = []
for xv in xs:
    x = torch.tensor(float(xv), requires_grad=True)
    y = x**3 + 2*x
    y.backward()
    autograd.append(x.grad.item())
autograd = np.array(autograd)

print("max abs difference:", np.max(np.abs(analytic - autograd)))  # ~0.0
for xv, a, g in zip(xs[::5], analytic[::5], autograd[::5]):
    print(f"x={xv:+.1f}  analytic={a:6.2f}  autograd={g:6.2f}")


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a leaf tensor w = torch.tensor([1.0, 2.0], requires_grad=True). Inside a training loop you compute a loss and call loss.backward() every iteration, but you forgot to zero the gradient. After 3 identical iterations, how does w.grad compare to a single iteration's gradient?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that .backward() accumulates into .grad rather than replacing it. — _This is the default so multiple backward passes can be summed (gradient accumulation)._
- Each iteration adds the same gradient g into w.grad. — _The forward is identical, so each backward deposits the same g._

**Answer:** w.grad equals 3&times; the single-iteration gradient. The fix is optimizer.zero_grad() (or w.grad.zero_()) at the top of each iteration so each step sees only its own gradient.

</details>

**Problem 2.** For x = torch.tensor(3.0, requires_grad=True) and y = x**2 + 5*x, what is x.grad after y.backward()?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Differentiate: $\frac{dy}{dx} = 2x + 5$. — _Power rule on $x^2$ gives $2x$; the $5x$ term gives $5$. Autograd applies the same chain rule automatically._
- Evaluate at $x=3$: $2(3)+5 = 11$. — _Plug the leaf value into the derivative._

**Answer:** x.grad is 11.0. PyTorch's autograd returns exactly this analytic value.

</details>

**Problem 3.** You wrap your evaluation loop in with torch.no_grad():. What changes about the tensors produced inside that block, and why do it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Inside the block, operations are not tracked: outputs have requires_grad=False and no grad_fn. — _no_grad turns off graph construction._
- No graph means no stored activations for a backward pass. — _At inference you never call .backward(), so that memory and bookkeeping is pure waste._

**Answer:** Tensors come out detached from the graph (requires_grad=False). You do it to save memory and speed up inference / evaluation, where gradients are not needed. (Closely related: .detach() cuts a single tensor out of the graph.)

</details>